In [6]:
import os
import torch

In [3]:
# load plain text of all Harry Potter books
with open('./source/Complete_Harry_Potter_txt_file/Harry_Potter_complete_dataset.txt', 'r', encoding='utf-8') as f:
    text = f.read()

print("Length of plain text in characters: ", len(text))

Length of plain text in characters:  6285613


In [4]:
# construct a vocabulary
chars = sorted(list(set(text)))
vocab_size = len(chars)
print(''.join(chars))
print(vocab_size)


 !"&'()*,-./0123456789:;<=>?ABCDEFGHIJKLMNOPQRSTUVWXYZ[\]^_`abcdefghijklmnopqrstuvwxyz{|}£¦«°»éü˜–—‘’“”•…
106


In [ ]:
# mapping the characters to integers (this is a typical progress called tokenizer)
# define a simple encoder and a decoder
stoi = { ch:i for i,ch in enumerate(chars)}
itos = { i:ch for i,ch in enumerate(chars)}
encode = lambda s: [stoi[c] for c in s]
decode = lambda l: ''.join(itos[i] for i in l)

print(encode('tell some harry potter jokes?'))
print(decode(encode('tell some harry potter jokes?')))

[80, 65, 72, 72, 1, 79, 75, 73, 65, 1, 68, 61, 78, 78, 85, 1, 76, 75, 80, 80, 65, 78, 1, 70, 75, 71, 65, 79, 28]
tell some harry potter jokes?


In [ ]:
# turn vocabulary list to tensor for parallel computing
data = torch.tensor(encode(text), dtype=torch.long)
print(data.shape, data.type)
print(data[:1000])

torch.Size([6285613]) <built-in method type of Tensor object at 0x11797d1d0>
tensor([ 41,   1,  78,  11,   1,  61,  74,  64,   1,  41,  78,  79,  11,   1,
         32,  81,  78,  79,  72,  65,  85,   9,   1,  75,  66,   1,  74,  81,
         73,  62,  65,  78,   1,  66,  75,  81,  78,   9,   1,  44,  78,  69,
         82,  65,  80,   1,  32,  78,  69,  82,  65,   9,   1,  83,  65,  78,
         65,   1,  76,  78,  75,  81,  64,   1,  80,  75,   1,  79,  61,  85,
          1,  80,  68,  61,  80,   1,  80,  68,  65,  85,   1,  83,  65,  78,
         65,   1,  76,  65,  78,  66,  65,  63,  80,  72,  85,   1,  74,  75,
         78,  73,  61,  72,   9,   1,  80,  68,  61,  74,  71,   1,  85,  75,
         81,   1,  82,  65,  78,  85,   1,  73,  81,  63,  68,  11,   1,  48,
         68,  65,  85,   1,  83,  65,  78,  65,   1,  80,  68,  65,   1,  72,
         61,  79,  80,   1,  76,  65,  75,  76,  72,  65,   1,  85,  75,  81,
        101,  64,   1,  65,  84,  76,  65,  63,  80,   1,  80,  7

In [8]:
# split the data into train sets and validation sets
n = int(0.9 * len(data))
train_data = data[:n]
val_data = data[n:]

In [ ]:
# chunk is a slice of the data
# block typically a unit that model can process, so block_size also can be recogenized as context length
block_size = 8

# clip an example of input and expected output in Timne Dimension
x = train_data[:block_size]
y = train_data[1:block_size+1]
for t in range(block_size):
    context = x[:t+1]
    target = y[t]
    print(f"Expected results when input is {context}, the target is {target}.")

# the reason why input length from one to block_size is that 
# we tend to make network can work with any length of the input


Expected results when input is tensor([41]), the target is 1.
Expected results when input is tensor([41,  1]), the target is 78.
Expected results when input is tensor([41,  1, 78]), the target is 11.
Expected results when input is tensor([41,  1, 78, 11]), the target is 1.
Expected results when input is tensor([41,  1, 78, 11,  1]), the target is 61.
Expected results when input is tensor([41,  1, 78, 11,  1, 61]), the target is 74.
Expected results when input is tensor([41,  1, 78, 11,  1, 61, 74]), the target is 64.
Expected results when input is tensor([41,  1, 78, 11,  1, 61, 74, 64]), the target is 1.


In [11]:
torch.manual_seed(507)

# Batch Dimension. the independent sequences transformer process (forward and backward) in parallel
# Aim to improve efficiency by parallel computing
batch_size = 4
block_size = 8

def get_batch(split):
    # randomly select a batch from the data
    data = train_data if split == 'train' else val_data
    # {batch_size} random numbers to locate the begin of the block
    ix = torch.randint(len(data) - block_size, (batch_size,))
    # stack: from vectors to a metrix
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    return x, y

xb, yb = get_batch('train')


In [ ]:
import torch.nn as nn
from torch.nn import functional as F
torch.manual_seed(507)

# 二元语言模型：
# 1.预测的时候仅根据上一个 token 的内容进行预测；
# 2.特征向量维数等于词汇表长度，词嵌入向量直接当做 logits 来用；
class BigramLanguageModel(nn.Module):
    
    def __init__(self, vocab_size):
        super().__init__()
        # 词嵌入，特征维数等于词汇表长度
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, idx, targets=None):
        logits = self.token_embedding_table(idx)
        
        if targets is None:
            loss = None
        else:
            # reshape to the expected input of cross_entropy
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            # 用交叉熵（ -ln(dist) ）作为最经典的 loss 函数
            loss = F.cross_entropy(logits, targets)

        return logits, loss
    
    # predict the next token and cat at the end of the input
    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            # call itself to predict
            logits, loss = self(idx)
            # only last token is the prediction we need
            logits = logits[:, -1, :]
            probs = F.softmax(logits, dim=-1)
            # pick up the specific token accroding to the probability
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx
    
m = BigramLanguageModel(vocab_size)
logits, loss = m(xb, yb)
print(logits.shape)
print(loss)

# 从一个换行符开始预测
print(decode(m.generate(idx=torch.zeros((1,1), dtype=torch.long), max_new_tokens=100)[0].tolist()))

torch.Size([32, 106])
tensor(5.1104, grad_fn=<NllLossBackward0>)

jkRp_G»=eq64bi0Bkz{Mmk03O–JrS…S•[| B:S˜A?w>küm‘Y
AYE»BjX]f•/–R/Xq`R3\éq7qc\6H;*L<]F–Ro«D}C}2'H"5B˜<]
